In [1]:
import pandas as pd
import random
import time
from pathlib import Path

In [2]:
file_path = Path("../data/munchee_optimization_dataset.xlsx")

products_df = pd.read_excel(file_path, sheet_name="Products")
inventory_df = pd.read_excel(file_path, sheet_name="Inventory")
demand_df = pd.read_excel(file_path, sheet_name="Weekly_Demand")
shops_df = pd.read_excel(file_path, sheet_name="Shops")
vehicles_df = pd.read_excel(file_path, sheet_name="Vehicles")

for df in [products_df, inventory_df, demand_df, shops_df, vehicles_df]:
    df.columns = df.columns.str.strip().str.lower()

products = products_df["product_id"].tolist()
shops = shops_df["shop_id"].tolist()
vehicles = vehicles_df["vehicle_id"].tolist()

opening_stock = dict(zip(inventory_df["product_id"], inventory_df["opening_stock_cartons"]))
safety_stock = dict(zip(inventory_df["product_id"], inventory_df["safety_stock_cartons"]))
usable_stock = {p: max(opening_stock[p] - safety_stock[p], 0) for p in products}

weights = dict(zip(products_df["product_id"], products_df["carton_weight_kg"]))
volumes = dict(zip(products_df["product_id"], products_df["carton_volume_m3"]))
margins = dict(zip(products_df["product_id"], products_df["gross_margin_lkr"]))

vehicle_carton_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_cartons"]))
vehicle_weight_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_payload_kg"]))
vehicle_volume_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_volume_m3"]))

total_carton_capacity = sum(vehicle_carton_capacity.values())
total_weight_capacity = sum(vehicle_weight_capacity.values())
total_volume_capacity = sum(vehicle_volume_capacity.values())

demand = {}
priority = {}

for _, row in demand_df.iterrows():
    demand[(row["shop_id"], row["product_id"])] = row["weekly_demand_cartons"]
    priority[(row["shop_id"], row["product_id"])] = row["priority_rank"]

In [3]:
decision_pairs = [(s, p) for s in shops for p in products]
num_genes = len(decision_pairs)
print(num_genes)

500


In [4]:
def create_individual():
    individual = []
    for s, p in decision_pairs:
        max_qty = demand[(s, p)]
        individual.append(random.randint(0, max_qty))
    return individual

In [5]:
def repair(individual):
    stock_used = {p: 0 for p in products}

    for i, qty in enumerate(individual):
        s, p = decision_pairs[i]
        stock_used[p] += qty

    for p in products:
        while stock_used[p] > usable_stock[p]:
            candidate_indexes = [i for i, (s, prod) in enumerate(decision_pairs) if prod == p and individual[i] > 0]
            if not candidate_indexes:
                break
            idx = random.choice(candidate_indexes)
            individual[idx] -= 1
            stock_used[p] -= 1

    while sum(individual) > total_carton_capacity:
        positive_indexes = [i for i, qty in enumerate(individual) if qty > 0]
        if not positive_indexes:
            break
        idx = random.choice(positive_indexes)
        individual[idx] -= 1

    while sum(weights[decision_pairs[i][1]] * individual[i] for i in range(num_genes)) > total_weight_capacity:
        positive_indexes = [i for i, qty in enumerate(individual) if qty > 0]
        if not positive_indexes:
            break
        idx = random.choice(positive_indexes)
        individual[idx] -= 1

    while sum(volumes[decision_pairs[i][1]] * individual[i] for i in range(num_genes)) > total_volume_capacity:
        positive_indexes = [i for i, qty in enumerate(individual) if qty > 0]
        if not positive_indexes:
            break
        idx = random.choice(positive_indexes)
        individual[idx] -= 1

    return individual

In [6]:
def fitness(individual):
    total_score = 0

    for i, qty in enumerate(individual):
        s, p = decision_pairs[i]
        score = ((6 - priority[(s, p)]) * 10 + margins[p]) * qty
        total_score += score

    return total_score

In [7]:
def tournament_selection(population, fitnesses, k=3):
    selected = random.sample(list(zip(population, fitnesses)), k)
    selected.sort(key=lambda x: x[1], reverse=True)
    return selected[0][0][:]

In [8]:
def crossover(parent1, parent2):
    point = random.randint(1, num_genes - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

In [9]:
def mutate(individual, mutation_rate=0.02):
    for i in range(num_genes):
        if random.random() < mutation_rate:
            s, p = decision_pairs[i]
            individual[i] = random.randint(0, demand[(s, p)])
    return individual

In [10]:
def run_ga(pop_size=60, generations=120, mutation_rate=0.03):
    population = [repair(create_individual()) for _ in range(pop_size)]

    best_solution = None
    best_fitness = float("-inf")

    for gen in range(generations):
        fitnesses = [fitness(ind) for ind in population]

        for ind, fit in zip(population, fitnesses):
            if fit > best_fitness:
                best_fitness = fit
                best_solution = ind[:]

        new_population = []

        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)

            child1, child2 = crossover(parent1, parent2)
            child1 = repair(mutate(child1, mutation_rate))
            child2 = repair(mutate(child2, mutation_rate))

            new_population.extend([child1, child2])

        population = new_population[:pop_size]

    return best_solution, best_fitness

In [11]:
start = time.time()
best_solution, best_fit = run_ga(pop_size=60, generations=150, mutation_rate=0.03)
end = time.time()

runtime_ga = end - start

print("Best Fitness:", best_fit)
print("Runtime:", round(runtime_ga, 4), "seconds")

Best Fitness: 758040
Runtime: 10.7373 seconds


In [12]:
ga_rows = []

for i, qty in enumerate(best_solution):
    if qty > 0:
        s, p = decision_pairs[i]
        ga_rows.append({
            "shop_id": s,
            "product_id": p,
            "allocated_cartons": qty
        })

ga_alloc_df = pd.DataFrame(ga_rows)
ga_alloc_df.head()

,shop_id,product_id,allocated_cartons
0,S001,P02,3
1,S001,P03,10
2,S001,P04,17
3,S001,P05,8
4,S001,P06,1


In [13]:
Path("../results/tables").mkdir(parents=True, exist_ok=True)
ga_alloc_df.to_csv("../results/tables/ga_allocation_results.csv", index=False)

In [14]:
total_demand = sum(demand.values())
fulfilled_demand_ga = ga_alloc_df["allocated_cartons"].sum()
unmet_demand_ga = total_demand - fulfilled_demand_ga
fill_rate_ga = (fulfilled_demand_ga / total_demand) * 100

used_stock_by_product_ga = ga_alloc_df.groupby("product_id")["allocated_cartons"].sum().to_dict()
used_stock_total_ga = sum(used_stock_by_product_ga.values())
total_usable_stock = sum(usable_stock.values())
stock_utilization_ga = (used_stock_total_ga / total_usable_stock) * 100 if total_usable_stock > 0 else 0

print("Total Demand:", total_demand)
print("Fulfilled Demand:", fulfilled_demand_ga)
print("Unmet Demand:", unmet_demand_ga)
print("Fill Rate %:", round(fill_rate_ga, 2))
print("Stock Utilization %:", round(stock_utilization_ga, 2))

Total Demand: 4082
Fulfilled Demand: 1440
Unmet Demand: 2642
Fill Rate %: 35.28
Stock Utilization %: 43.33
